# 02 — Reference Panel Check

## What this notebook does
Runs a small calibration panel — a positive reference, an optional weaker control,
a scrambled-sequence negative, and a poly-Ala nonsense control — and checks whether
the scoring lane can tell them apart at all.

## What decision it helps make
> "The lane separates my reference from obvious nonsense. I am ready to score real candidates."

OR

> "The lane cannot distinguish signal from noise. I must fix the setup before proceeding."

## What it cannot prove
- That the scoring lane is physically correct
- That a 'pass' here means your candidates will rank correctly
- Anything about binding affinity

---

> **Key concept: What is a 'lane'?**
>
> A **lane** is a specific combination of tools and parameters used to score or evaluate
> peptides. Different lanes (heuristic, docking, structure-prediction) give different
> scores. A lane can only be trusted for ranking if it first passes calibration.
>
> **What is a 'reference panel'?**
>
> A set of peptides with known expected behavior: a real positive control that *should*
> score well, and negative controls that *should* score poorly. Running them through
> your lane tells you whether the lane is doing anything useful.

## Free vs Paid Colab

This notebook runs fully on **free Colab**. The heuristic lane has no GPU requirement.

| Step | Free | Paid |
|------|------|------|
| Load target spec | Yes | Yes |
| Run heuristic panel (4 peptides) | Yes | Yes |
| Run larger panel (up to 20) | Yes | Yes |

If `COMPUTE_TIER = "paid"`, the panel size cap is raised. The heuristic lane itself
does not change — only the maximum number of control peptides you can include.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "biopython", "matplotlib"])
print("Dependencies ready.")

In [ ]:
import sys, pathlib

# ── Environment detection ─────────────────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    for _p in [
        pathlib.Path('/content/peptide-cookbooks-internal/colab-basics'),
        pathlib.Path('/content/colab-basics'),
    ]:
        if _p.exists():
            COOKBOOK_DIR = _p
            break
    else:
        raise RuntimeError(
            "Cookbook not found. Clone the repo first:\n"
            "  !git clone <your-repo-url> /content/peptide-cookbooks-internal"
        )
    WORKSPACE_DIR = pathlib.Path('/content/workspace')
else:
    COOKBOOK_DIR = pathlib.Path('..').resolve()
    WORKSPACE_DIR = COOKBOOK_DIR / 'workspace'

# ============================================================
# CONFIGURATION
# ============================================================

# Compute tier — controls panel size cap only. No GPU lane in this cookbook.
COMPUTE_TIER = "free"  # "free" or "paid"

TARGET_SPEC_PATH = WORKSPACE_DIR / "target_spec.json"
OUTPUT_DIR = WORKSPACE_DIR / "reference_panel"

# ── Optional extra controls ──────────────────────────────────────────────────
WEAK_POSITIVE_SEQUENCE = None   # e.g. "ACRKLNRSFM" — optional weak/proxy positive
WEAK_POSITIVE_LABEL = "weak_positive"
CUSTOM_SCRAMBLED = None         # leave None to auto-generate from positive
POLY_ALA_LENGTH = 12            # roughly match the length of your reference

MAX_PANEL_SIZE = 20 if COMPUTE_TIER == "paid" else 8

print(f"Environment: {'Colab' if IN_COLAB else 'local'}, COMPUTE_TIER: {COMPUTE_TIER}")
print(f"Target spec: {TARGET_SPEC_PATH}")

In [ ]:
# Setup paths and imports
import sys, pathlib, json

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(COOKBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(COOKBOOK_DIR))

from shared.target_utils import load_target_spec
from shared.scoring_utils import score_panel, interpret_panel_separation
from shared.panel_utils import make_reference_panel, save_panel_results, format_panel_table

# Load the frozen target spec from notebook 01
if not TARGET_SPEC_PATH.exists():
    raise FileNotFoundError(
        f"Target spec not found at {TARGET_SPEC_PATH}. "
        "Run notebook 01 first and ensure the workspace path matches."
    )

target_spec = load_target_spec(TARGET_SPEC_PATH)
ref_sequence = target_spec["reference_policy"]["positive_control"]["sequence"]
print(f"Loaded target spec for PDB {target_spec['pdb_id']}, chain {target_spec['chain_id']}")
print(f"Positive reference: {ref_sequence}")

## Step 1 — Build the Reference Panel

The panel contains four types of control:

| Role | Purpose |
|------|---------|
| `positive_control` | A real or well-motivated benchmark peptide. Should score highest. |
| `weak_positive` | A weaker or proxy binder (optional). Should score between positive and negatives. |
| `negative_control` | Scrambled-sequence control. Same amino acid composition, shuffled order. |
| `nonsense_control` | Poly-Alanine. If this passes, the lane is blind to sequence specificity. |

> **Why poly-Alanine?**
> Alanine is small, uncharged, and non-polar. A poly-Ala peptide has almost no specific
> properties that would make it fit any particular pocket. If your lane scores it as well
> as a real positive, the lane is not detecting anything sequence-specific.

In [ ]:
panel = make_reference_panel(
    positive_control=ref_sequence,
    positive_control_label="primary_positive",
    weak_positive=WEAK_POSITIVE_SEQUENCE,
    weak_positive_label=WEAK_POSITIVE_LABEL,
    scrambled_control=CUSTOM_SCRAMBLED,
    poly_ala_length=POLY_ALA_LENGTH,
)

print(f"Panel has {len(panel)} entries:")
for entry in panel:
    print(f"  [{entry['role']:<20}] {entry['label']:<25} {entry['sequence']}")

## Step 2 — Score the Panel

The scoring lane here is a **composition and property heuristic**. It is not docking
and not a physics engine. It evaluates:
- Whether the peptide length fits the pocket hints
- Whether the charge complements the pocket hints
- Whether the hydrophobic content matches the pocket hints
- How similar the sequence is to the positive reference (BLOSUM62)

This lane can separate a reference from a completely wrong peptide.
It cannot rank fine differences between similar sequences reliably.
All four sub-scores use the pocket hints from `target_spec.json`.

In [ ]:
results = score_panel(panel, target_spec)

print("Scored panel (ranked by composite score):")
print(format_panel_table(results))

## Step 3 — Interpret the Separation

In [ ]:
status, explanation = interpret_panel_separation(results)

status_emoji = {"pass": "PASS", "degraded": "DEGRADED", "fail": "FAIL", "unknown": "UNKNOWN"}
print(f"\n=== Panel Status: {status_emoji.get(status, status)} ===")
print(explanation)

if status == "fail":
    print("\nDo NOT proceed to notebooks 03 or 04 until this is resolved.")
    print()
    print("RECOVERY STEPS — try in this order:")
    print()
    print("  Step A — Check POCKET_HYDROPHOBIC_HINT first (most common cause).")
    print("    The poly-Ala control has a hydrophobic fraction of ~0.62.")
    print("    If your hint is between 0.50 and 0.70, poly-Ala will match it as well as")
    print("    or better than most real peptides. Look up your pocket in a paper or")
    print("    use the pocket residue list from Step 3 — if the residues are mostly")
    print("    charged (K, R, D, E) or polar (N, Q, S, T), set the hint to 0.2–0.4.")
    print("    If the pocket is genuinely hydrophobic, set it to 0.65–0.85.")
    print()
    print("  Step B — Check POCKET_NET_CHARGE_HINT.")
    print("    Poly-Ala is neutral (charge 0). A charged pocket (hint ≥ 1.5 or ≤ -1.5)")
    print("    will reward charged peptides and penalise poly-Ala. If your pocket is")
    print("    clearly charged but you set hint=0.0, fix it — look at the K/R/D/E residues")
    print("    in your pocket residue list.")
    print()
    print("  Step C — Verify your positive reference is actually plausible.")
    print("    If the reference is too short, too neutral, or too hydrophilic for the")
    print("    pocket you described, it will lose to poly-Ala on the hydrophobic sub-score.")
    print("    Try a different reference sequence, or extend it by 2–4 residues.")
    print()
    print("  Step D — Check POCKET_RESIDUE_IDS and TARGET_CHAIN_ID in notebook 01.")
    print("    If you selected residues from a ligand chain or a crystal contact, the")
    print("    pocket hints won't describe a real binding surface.")
    print()
    print("  After any change: update notebook 01 config → re-run notebook 01 → re-run")
    print("  this notebook. Repeat until status is 'pass' or 'degraded'.")

elif status == "degraded":
    print("\nYou may proceed but be cautious. Fine-grained ranking may not be reliable.")
    print("If you want a cleaner signal:")
    print("  - Tighten POCKET_HYDROPHOBIC_HINT toward the actual pocket character")
    print("  - Add a WEAK_POSITIVE_SEQUENCE if you have a weaker known binder")
    print("  - Increase POCKET_NET_CHARGE_HINT if the pocket is charged")
    print("Fine-grained SAR distinctions (notebook 04) may not be trustworthy at this separation level.")

else:
    print("\nYou may proceed to notebook 03.")

## Step 4 — Plot the Panel

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

labels = [r['label'] for r in results]
scores = [float(r.get('composite_score', 0)) for r in results]
roles = [r.get('role', 'unknown') for r in results]

color_map = {
    "positive_control": "#2ecc71",
    "negative_control": "#e74c3c",
    "nonsense_control": "#c0392b",
    "unknown": "#95a5a6",
}
colors = [color_map.get(r, "#95a5a6") for r in roles]

fig, ax = plt.subplots(figsize=(max(6, len(labels) * 1.2), 4))
bars = ax.bar(range(len(labels)), scores, color=colors, edgecolor='white', linewidth=0.8)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=35, ha='right', fontsize=9)
ax.set_ylabel("Composite Heuristic Score", fontsize=10)
ax.set_title(f"Reference Panel — Heuristic Lane\n{target_spec['pdb_id']} chain {target_spec['chain_id']}",
             fontsize=11)
ax.set_ylim(0, 1.0)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.5, label='midpoint')

legend_patches = [
    mpatches.Patch(color='#2ecc71', label='positive control'),
    mpatches.Patch(color='#e74c3c', label='negative / nonsense control'),
]
ax.legend(handles=legend_patches, fontsize=8)

# Add score labels on bars
for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{score:.3f}", ha='center', va='bottom', fontsize=8)

plt.tight_layout()
chart_path = OUTPUT_DIR / "panel_chart.png"
plt.savefig(chart_path, dpi=120, bbox_inches='tight')
plt.show()
print(f"Chart saved to {chart_path}")

In [ ]:
# Save scored panel to CSV and interpretation to text
from shared.panel_utils import save_panel_results

csv_path = OUTPUT_DIR / "panel_scores.csv"
save_panel_results(results, csv_path)
print(f"Scores saved to {csv_path}")

interp_path = OUTPUT_DIR / "panel_interpretation.txt"
interp_text = (
    f"Panel status: {status}\n\n"
    f"{explanation}\n\n"
    f"Lane: heuristic_composition\n"
    f"Target: {target_spec['pdb_id']} chain {target_spec['chain_id']}\n"
    f"Positive reference: {ref_sequence}\n"
)
interp_path.write_text(interp_text)
print(f"Interpretation saved to {interp_path}")

## Understanding Pass / Degraded / Fail

| Status | Meaning | What to do |
|--------|---------|------------|
| **pass** | Reference clearly above controls. Lane can support rough triage. | Proceed to notebook 03. |
| **degraded** | Weak separation. Ranking is noisy. | Adjust pocket hints, then decide. |
| **fail** | No separation. Lane cannot distinguish signal from noise. | Do not proceed. Fix setup first. |

### What 'fail' actually means

A failing panel means the scoring lane (as configured) is **sequence-insensitive** for
this target. The nonsense controls are surviving as well as the reference. This means any
ranking you do in notebook 03 or 04 is meaningless — you'd just be measuring noise.

Common causes:
- `POCKET_NET_CHARGE_HINT` or `POCKET_HYDROPHOBIC_HINT` are wrong for this target
- Positive reference is not actually a good binder (or is too similar to poly-Ala)
- The target structure is not the right conformation for your hypothesis
- The heuristic lane is not the right tool for this particular target

In the last case, you need a stronger lane (docking or structure prediction).
See `NOTEBOOK_PLAN.md` for a discussion of this limitation.

## Outputs

| File | Description |
|------|-------------|
| `workspace/reference_panel/panel_scores.csv` | Scored panel with all sub-scores |
| `workspace/reference_panel/panel_chart.png` | Bar chart of panel scores |
| `workspace/reference_panel/panel_interpretation.txt` | Plain-text pass/fail summary |

## Next Notebook

→ **03_single_peptide_eval.ipynb** (only if panel status is 'pass' or 'degraded')

If the panel failed, revisit notebook 01 and adjust your pocket hints or reference peptide.